## Calibrate NCHR BPR Data

Loads the multi-year raw frequency dataset from `out/NCHR_BPR_raw.parquet/`,
then applies Paroscientific and Platinum RTD calibration to produce physical
pressure (dbar) and temperature (°C) time-series.

> **Kernel:** select the `bpr-nchr` conda environment.

In [ ]:
import pandas as pd
import numpy as np
from src.calibrateBPRData import CalibrationCoefficients as CCD
from src.build_parquet import load_dataset

pd.set_option('display.precision', 15)

In [ ]:
# Sensor serial numbers / IDs for the NCHR deployment
paroSensorID = 93996   # Paroscientific pressure + seawater-temp sensor
loggerID     = 0xB9    # BPR logger ID
TempSensorID = 0x98    # Platinum RTD housing-temperature sensor

ccd = CCD()
CP_SF = ccd.getParoCoeffs(paroSensorID)      # Paroscientific calibration coefficients
CT_Ti = ccd.getPlatinumCoeffs(TempSensorID)  # Platinum RTD calibration coefficients

print('Paroscientific coefficients:', CP_SF)
print('Platinum coefficients      :', CT_Ti)

### Load the raw frequency dataset

In [ ]:
# Load full dataset — restrict with date_from / date_to for a subset
df = load_dataset()
# df = load_dataset(date_from='2025-01-01', date_to='2025-02-01')

print(f'Rows loaded : {len(df):,}')
print(f'Date range  : {df.index.min()}  →  {df.index.max()}')
df.head()

In [ ]:
df.dtypes

### Modify your calibration coefficients

In [ ]:
# current coefficients
CP_SF

In [ ]:
CP_SF['U0'] = 5.81266
CP_SF['D1'] = 0.14432

### Calibrate: counts / periods → physical units

In [ ]:
# ── Vectorized calibration (operates on entire columns at once) ──────────────

# --- Housing temperature (Platinum RTD): T = a·x + b ---
df['T_Housing'] = CT_Ti['a'] * df['t_housing_counts'] + CT_Ti['b']

# --- Seawater temperature (Paroscientific Type-II) ---
# U = ((xFT + 2^32) * 4.656612873e-9 / 4) - U0
xFT = df['xFT'].to_numpy(dtype=float)
U_T = (xFT + 4_294_967_296) * 4.656612873e-9 / 4 - CP_SF['U0']
U_T[xFT == 0] = np.nan          # sentinel: period=0 → NaN
df['T_SF'] = CP_SF['Y1'] * U_T + CP_SF['Y2'] * U_T**2

# --- Pressure (dbar), temperature-compensated ---
# Invert T_SF → compensation frequency U_P for Type-II:
#   U = -(Y1 + sqrt(Y1² + 4·Y2·T)) / (2·Y2)
T_arr = df['T_SF'].to_numpy(dtype=float)
U_P   = -(CP_SF['Y1'] + np.sqrt(CP_SF['Y1']**2 + 4 * CP_SF['Y2'] * T_arr)) / (2 * CP_SF['Y2'])

xFP    = df['xFP'].to_numpy(dtype=float)
Tper   = (xFP + 4_294_967_296) * 4.656612873e-9          # pressure period
CU     = CP_SF['C1'] + CP_SF['C2'] * U_P + CP_SF['C3'] * U_P**2
T0     = CP_SF['T1'] + CP_SF['T2'] * U_P + CP_SF['T3'] * U_P**2 + CP_SF['T4'] * U_P**3
ratio  = (T0 / Tper)**2
P_dbar = CU * (1 - ratio) * (1 - CP_SF['D1'] * (1 - ratio)) * 6.894757 / 10

# Mask invalid rows
P_dbar[xFP == 0]        = np.nan
P_dbar[np.isnan(T_arr)] = np.nan
df['P_SF'] = P_dbar

In [ ]:
df[['T_Housing', 'T_SF', 'P_SF']].describe()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

df['P_SF'].plot(ax=axes[0], lw=0.5, color='steelblue')
axes[0].set_ylabel('Pressure (dbar)')

df['T_SF'].plot(ax=axes[1], lw=0.5, color='tomato')
axes[1].set_ylabel('Seawater Temp (°C)')

df['T_Housing'].plot(ax=axes[2], lw=0.5, color='seagreen')
axes[2].set_ylabel('Housing Temp (°C)')

axes[0].set_title('NCHR BPR – Calibrated time series')
plt.tight_layout()
plt.show()

### Optional: export calibrated data to CSV

In [ ]:
# Uncomment to save the calibrated output
out_path = 'out/NCHR_calibrated_new.csv'
df[['ppc_time', 'T_Housing', 'T_SF', 'P_SF']].to_csv(out_path)
print(f'Saved to {out_path}')